# xLSTM + Recurrent PPO: торговля на MOEX (улучшенная версия)

Улучшения по сравнению с предыдущей версией:
- Нормализация только по train (нет look-ahead bias)
- Технические индикаторы: RSI, MACD, BB, ATR, ADX, OBV, MFI, EMA trend, Momentum
- Улучшенный reward: бонус за правильное направление + штраф за плохой шорт
- Автоматическое закрытие шорта при развороте рынка
- margin_ratio=0.25 (было 0.5)
- Curriculum learning: сначала спокойный рынок, потом кризисы
- 1M шагов обучения
- n_steps=2048, ent_coef=0.02, lr=1e-4, clip_range=0.1

In [ ]:
!pip install stable-baselines3 sb3-contrib pandas numpy requests apimoex matplotlib pandas_ta -q
!pip install git+https://github.com/NX-AI/xlstm.git -q

In [ ]:
import pandas as pd
import numpy as np
import requests
import apimoex
import pandas_ta as ta
import warnings
warnings.filterwarnings('ignore')

## Шаг 1: Загрузка данных

In [ ]:
TICKERS     = ['SBER', 'GAZP', 'LKOH', 'YNDX', 'GMKN']
TRAIN_START = '2009-01-01'
TRAIN_END   = '2021-12-31'
TEST_START  = '2022-01-01'
TEST_END    = '2024-01-01'
DATA_FILE   = 'moex_data.csv'


def download_moex_ohlcv(ticker, start, end):
    with requests.Session() as session:
        data = apimoex.get_board_history(
            session, ticker, start=start, end=end,
            columns=('TRADEDATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME'))
    if not data:
        raise ValueError(f'Нет данных для {ticker}')
    df = pd.DataFrame(data)
    df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
    df = df.set_index('TRADEDATE')
    df.columns = [c.lower() for c in df.columns]
    df = df.add_prefix(f'{ticker}_')
    return df.replace(0, np.nan)


def load_or_download(tickers, start, end, cache_file):
    import os
    if os.path.exists(cache_file):
        print(f'Загружаем из кэша: {cache_file}')
        return pd.read_csv(cache_file, index_col=0, parse_dates=True)
    print('Скачиваем данные с MOEX ISS...')
    frames = []
    for t in tickers:
        print(f'  {t}...', end=' ', flush=True)
        frames.append(download_moex_ohlcv(t, start, end))
        print('OK')
    df = pd.concat(frames, axis=1).ffill().bfill()
    df.to_csv(cache_file)
    print(f'Сохранено в {cache_file}')
    return df


raw = load_or_download(TICKERS, TRAIN_START, TEST_END, DATA_FILE)
print(f'Загружено: {raw.shape[0]} дней, {raw.shape[1]} колонок')
print(raw.head())

## Шаг 2: Features + Turbulence (с исправлением look-ahead bias)

In [ ]:
def extract_close_prices(df, tickers):
    return df[[f'{t}_close' for t in tickers]].copy()


def compute_turbulence(close_prices, lookback=252):
    returns    = close_prices.pct_change().dropna()
    turbulence = pd.Series(index=returns.index, dtype=float, name='turbulence')
    for i in range(lookback, len(returns)):
        hist = returns.iloc[i - lookback:i]
        y    = returns.iloc[i].values - hist.mean().values
        try:
            turb = float(y @ np.linalg.pinv(hist.cov().values) @ y)
        except Exception:
            turb = 0.0
        turbulence.iloc[i] = max(turb, 0.0)
    return turbulence.fillna(0.0)


def compute_features(df, tickers, fit_mask=None):
    """
    Вычисляет признаки с техническими индикаторами.
    fit_mask: булева маска обучающей выборки — нормализация ТОЛЬКО по ней.
    Это устраняет look-ahead bias.
    """
    parts = []
    for t in tickers:
        close  = df[f'{t}_close']
        high   = df[f'{t}_high']
        low    = df[f'{t}_low']
        volume = df[f'{t}_volume']

        # ── Базовые OHLCV ──────────────────────────────────────────────────────
        sub = df[[f'{t}_{c}' for c in ['open', 'high', 'low', 'close', 'volume']]].copy()

        # ── Тренд ──────────────────────────────────────────────────────────────
        # EMA cross — ключевой сигнал для long/short переключения
        sub[f'{t}_trend'] = (
            ta.ema(close, length=10) - ta.ema(close, length=30)
        ) / (close.abs() + 1e-8)

        sub[f'{t}_adx']   = ta.adx(high, low, close, length=14)['ADX_14'] / 100
        sub[f'{t}_rsi']   = ta.rsi(close, length=14) / 100

        # ── Моментум ───────────────────────────────────────────────────────────
        macd = ta.macd(close)
        sub[f'{t}_macd']  = macd['MACD_12_26_9'] / (close.abs() + 1e-8)
        sub[f'{t}_mom']   = ta.mom(close, length=10) / (close.abs() + 1e-8)

        # ── Волатильность ──────────────────────────────────────────────────────
        bb = ta.bbands(close, length=20)
        sub[f'{t}_bb_pct'] = (
            (close - bb['BBL_20_2.0']) /
            (bb['BBU_20_2.0'] - bb['BBL_20_2.0'] + 1e-8)
        )
        sub[f'{t}_atr']   = ta.atr(high, low, close, length=14) / (close.abs() + 1e-8)

        # ── Объём ──────────────────────────────────────────────────────────────
        sub[f'{t}_obv']   = ta.obv(close, volume)
        sub[f'{t}_mfi']   = ta.mfi(high, low, close, volume, length=14) / 100

        # Заполняем NaN которые появляются в начале из-за lookback индикаторов
        sub = sub.fillna(0)

        # ── Нормализация ТОЛЬКО по train (исправление look-ahead bias) ─────────
        if fit_mask is not None:
            mean = sub[fit_mask].mean()
            std  = sub[fit_mask].std() + 1e-8
        else:
            mean = sub.mean()
            std  = sub.std() + 1e-8

        sub = (sub - mean) / std
        parts.append(sub.values)

    return np.concatenate(parts, axis=1).astype(np.float32)


# ── Вычисляем всё ─────────────────────────────────────────────────────────────
close_prices = extract_close_prices(raw, TICKERS)

print('Считаем turbulence index (~1–2 мин)...')
turbulence = compute_turbulence(close_prices)

TURBULENCE_THRESHOLD = float(
    np.percentile(turbulence[TRAIN_START:TRAIN_END].dropna(), 75))
print(f'Turbulence threshold (75th pct): {TURBULENCE_THRESHOLD:.2f}')

common_idx   = raw.index.intersection(turbulence.dropna().index)
raw_aligned  = raw.loc[common_idx]
turb_aligned = turbulence.loc[common_idx]
dates_all    = raw_aligned.index

# Маски — нужны до вычисления features для правильной нормализации
train_mask = (dates_all >= TRAIN_START) & (dates_all <= TRAIN_END)
test_mask  = (dates_all >= TEST_START)  & (dates_all <= TEST_END)

print('Вычисляем технические индикаторы...')
# fit_mask=train_mask — нормализация только по обучающей выборке
features_all = compute_features(raw_aligned, TICKERS, fit_mask=train_mask)
close_all    = close_prices.loc[common_idx].values.astype(np.float32)

features_train = features_all[train_mask]
close_train    = close_all[train_mask]
turb_train_arr = turb_aligned[train_mask].values

features_test  = features_all[test_mask]
close_test     = close_all[test_mask]
turb_test_arr  = turb_aligned[test_mask].values

print(f'Train: {features_train.shape[0]} дней')
print(f'Test:  {features_test.shape[0]} дней')
print(f'Feature dim: {features_train.shape[1]}  (было 25, стало {features_train.shape[1]})')

# Проверка пропусков в тесте (торговля MOEX была приостановлена в марте 2022)
test_dates = dates_all[test_mask]
gaps = pd.Series(test_dates).diff().dt.days
large_gaps = gaps[gaps > 5]
if len(large_gaps) > 0:
    print(f'\nБольшие пропуски в тестовых данных (>5 дней):')
    print(large_gaps)
else:
    print('\nПропусков в тестовых данных нет.')

## Шаг 3: Торговые среды

In [ ]:
import gymnasium as gym
from gymnasium import spaces


class MOEXTradingEnv(gym.Env):
    """Long-only базовая среда."""
    metadata = {'render_modes': []}

    def __init__(self, close_prices, features, turbulence, turbulence_threshold,
                 initial_balance=1_000_000.0, window_size=30,
                 commission=0.0005, max_shares_per_trade=100):
        super().__init__()
        self.close_prices         = close_prices
        self.features             = features
        self.turbulence           = turbulence
        self.turbulence_threshold = turbulence_threshold
        self.initial_balance      = initial_balance
        self.window_size          = window_size
        self.commission           = commission
        self.max_shares           = max_shares_per_trade
        self.n_assets             = close_prices.shape[1]
        self.n_features           = features.shape[1]
        self.action_space      = spaces.Box(
            low=-1.0, high=1.0, shape=(self.n_assets,), dtype=np.float32)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(window_size, self.n_features), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step         = self.window_size
        self.balance              = self.initial_balance
        self.shares_held          = np.zeros(self.n_assets, dtype=np.float32)
        self.prev_portfolio_value = self.initial_balance
        self.portfolio_history    = [self.initial_balance]
        self.trade_count          = 0
        return self._get_obs(), {}

    def step(self, action):
        turb = self.turbulence[self.current_step]
        if turb > self.turbulence_threshold:
            self.current_step += 1
            done = self.current_step >= len(self.close_prices)
            self.portfolio_history.append(self._portfolio_value())
            return self._get_obs(), -1.0, done, False, {'turbulence': turb}

        prices       = self.close_prices[self.current_step]
        shares_delta = (action * self.max_shares).astype(np.float32)
        shares_delta = np.maximum(shares_delta, -self.shares_held)

        buy_cost = np.sum(np.maximum(shares_delta, 0) * prices)
        if buy_cost > self.balance:
            shares_delta = np.where(
                shares_delta > 0,
                shares_delta * self.balance / (buy_cost + 1e-8),
                shares_delta)

        cost             = np.sum(np.abs(shares_delta) * prices) * self.commission
        self.balance    -= np.sum(shares_delta * prices) + cost
        self.shares_held += shares_delta
        self.trade_count += int(np.any(shares_delta != 0))
        self.current_step += 1

        done      = self.current_step >= len(self.close_prices)
        new_value = self._portfolio_value()
        self.portfolio_history.append(new_value)
        reward    = float(np.clip(
            (new_value - self.prev_portfolio_value) / (self.prev_portfolio_value + 1e-8),
            -1.0, 1.0))
        self.prev_portfolio_value = new_value
        return self._get_obs(), reward, done, False, {
            'portfolio_value': new_value, 'turbulence': turb}

    def render(self):
        v = self._portfolio_value()
        print(f'День {self.current_step:4d} | {v:>14,.0f} ₽ | {(v/self.initial_balance-1)*100:+.2f}%')

    def _get_obs(self):
        return self.features[
            self.current_step - self.window_size:self.current_step].copy()

    def _portfolio_value(self):
        idx = min(self.current_step, len(self.close_prices) - 1)
        return float(self.balance + np.sum(
            self.shares_held * self.close_prices[idx]))


class MOEXTradingEnvShort(MOEXTradingEnv):
    """
    Среда с short selling + улучшенный reward + автозакрытие шорта.

    Улучшения:
    - margin_ratio=0.25 (снижен риск)
    - Бонус reward за правильное направление позиции
    - Штраф за удержание шорта на растущем рынке
    - Автоматическое закрытие шорта при росте цены >2%
    """

    def __init__(self, *args, margin_ratio=0.25, **kwargs):
        super().__init__(*args, **kwargs)
        self.margin_ratio = margin_ratio
        self._prev_prices = None

    def reset(self, seed=None, options=None):
        obs, info        = super().reset(seed=seed, options=options)
        self._prev_prices = None
        return obs, info

    def step(self, action):
        turb = self.turbulence[self.current_step]
        if turb > self.turbulence_threshold:
            self.current_step += 1
            done = self.current_step >= len(self.close_prices)
            self.portfolio_history.append(self._portfolio_value())
            self._prev_prices = None
            return self._get_obs(), -1.0, done, False, {'turbulence': turb}

        prices = self.close_prices[self.current_step]

        # ── Автоматическое закрытие шорта при развороте рынка ─────────────────
        if self._prev_prices is not None:
            price_change_auto = (
                (prices - self._prev_prices) / (self._prev_prices + 1e-8))
            for i in range(self.n_assets):
                if self.shares_held[i] < 0 and price_change_auto[i] > 0.02:
                    # Цена выросла >2% — принудительно закрываем шорт
                    close_delta      = -self.shares_held[i]
                    cost             = close_delta * prices[i] * (1 + self.commission)
                    self.balance    -= cost
                    self.shares_held[i] = 0

        self._prev_prices = prices.copy()

        # ── Ограничение на шорт ───────────────────────────────────────────────
        shares_delta    = (action * self.max_shares).astype(np.float32)
        max_short_value = self._portfolio_value() * self.margin_ratio

        for i in range(self.n_assets):
            if shares_delta[i] < 0:
                max_short_shares = max_short_value / (prices[i] + 1e-8)
                shares_delta[i]  = max(
                    shares_delta[i],
                    -max_short_shares - self.shares_held[i])

        # ── Ограничение баланса для покупок ───────────────────────────────────
        buy_cost = np.sum(np.maximum(shares_delta, 0) * prices)
        if buy_cost > self.balance:
            shares_delta = np.where(
                shares_delta > 0,
                shares_delta * self.balance / (buy_cost + 1e-8),
                shares_delta)

        cost             = np.sum(np.abs(shares_delta) * prices) * self.commission
        self.balance    -= np.sum(shares_delta * prices) + cost
        self.shares_held += shares_delta
        self.trade_count += int(np.any(shares_delta != 0))
        self.current_step += 1

        done      = self.current_step >= len(self.close_prices)
        new_value = self._portfolio_value()
        self.portfolio_history.append(new_value)

        # ── Улучшенный reward ─────────────────────────────────────────────────
        next_idx     = min(self.current_step, len(self.close_prices) - 1)
        prices_next  = self.close_prices[next_idx]
        price_change = (prices_next - prices) / (prices + 1e-8)

        # Базовый reward — изменение портфеля
        raw_reward = (
            (new_value - self.prev_portfolio_value) /
            (self.prev_portfolio_value + 1e-8))

        # Бонус: позиция совпадает с направлением рынка
        direction_bonus = np.sum(
            np.sign(self.shares_held) * price_change) * 0.1
        raw_reward += direction_bonus

        # Штраф: держим шорт на растущем рынке
        bad_short = np.sum(
            (self.shares_held < 0) *
            (price_change > 0) *
            np.abs(self.shares_held) * prices
        ) / (self.prev_portfolio_value + 1e-8)
        raw_reward -= bad_short * 0.2

        reward = float(np.clip(raw_reward, -1.0, 1.0))
        self.prev_portfolio_value = new_value

        return self._get_obs(), reward, done, False, {
            'portfolio_value': new_value,
            'balance': self.balance,
            'turbulence': turb,
        }

    def _portfolio_value(self):
        idx = min(self.current_step, len(self.close_prices) - 1)
        return float(self.balance + np.sum(
            self.shares_held * self.close_prices[idx]))


# ── Создаём все окружения ─────────────────────────────────────────────────────
WINDOW_SIZE = 30

# Long-only (для LSTM baseline)
env_train = MOEXTradingEnv(
    close_prices=close_train, features=features_train,
    turbulence=turb_train_arr, turbulence_threshold=TURBULENCE_THRESHOLD,
    window_size=WINDOW_SIZE)
env_test = MOEXTradingEnv(
    close_prices=close_test, features=features_test,
    turbulence=turb_test_arr, turbulence_threshold=TURBULENCE_THRESHOLD,
    window_size=WINDOW_SIZE)

# Short-enabled
env_train_short = MOEXTradingEnvShort(
    close_prices=close_train, features=features_train,
    turbulence=turb_train_arr, turbulence_threshold=TURBULENCE_THRESHOLD,
    window_size=WINDOW_SIZE, margin_ratio=0.25)
env_test_short = MOEXTradingEnvShort(
    close_prices=close_test, features=features_test,
    turbulence=turb_test_arr, turbulence_threshold=TURBULENCE_THRESHOLD,
    window_size=WINDOW_SIZE, margin_ratio=0.25)

# Быстрая проверка
obs, _ = env_train_short.reset()
obs2, reward, done, trunc, info = env_train_short.step(
    env_train_short.action_space.sample())
print(f'Short среда работает. Reward: {reward:.4f}')
print(f'Observation shape: {obs.shape}')
print(f'Action space: {env_train_short.action_space}  (< 0 = шорт)')

## Шаг 4: xLSTM Encoder

In [ ]:
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from sb3_contrib import RecurrentPPO
from xlstm import (
    xLSTMBlockStack, xLSTMBlockStackConfig,
    mLSTMBlockConfig, mLSTMLayerConfig,
    sLSTMBlockConfig, sLSTMLayerConfig, FeedForwardConfig,
)


def build_xlstm_stack(embedding_dim, context_length, num_blocks=4, use_slstm=False):
    if use_slstm:
        cfg = xLSTMBlockStackConfig(
            mlstm_block=mLSTMBlockConfig(
                mlstm=mLSTMLayerConfig(
                    conv1d_kernel_size=4, qkv_proj_blocksize=4, num_heads=4)),
            slstm_block=sLSTMBlockConfig(
                slstm=sLSTMLayerConfig(
                    backend='cuda', num_heads=4,
                    conv1d_kernel_size=4,
                    bias_init='powerlaw_blockdependent'),
                feedforward=FeedForwardConfig(proj_factor=1.3, act_fn='gelu')),
            context_length=context_length, num_blocks=num_blocks,
            embedding_dim=embedding_dim, slstm_at=[1])
    else:
        cfg = xLSTMBlockStackConfig(
            mlstm_block=mLSTMBlockConfig(
                mlstm=mLSTMLayerConfig(
                    conv1d_kernel_size=4, qkv_proj_blocksize=4, num_heads=4)),
            context_length=context_length, num_blocks=num_blocks,
            embedding_dim=embedding_dim, slstm_at=[])
    return xLSTMBlockStack(cfg)


class xLSTMEncoder(BaseFeaturesExtractor):
    def __init__(self, observation_space, embedding_dim=128,
                 num_blocks=4, use_slstm=False):
        super().__init__(observation_space, features_dim=embedding_dim)
        window_size, n_features = observation_space.shape
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, embedding_dim), nn.GELU())
        self.xlstm = build_xlstm_stack(
            embedding_dim, window_size, num_blocks, use_slstm)

    def forward(self, observations):
        x = self.input_proj(observations)
        x = self.xlstm(x)
        return x[:, -1, :]


def check_slstm_available():
    if not torch.cuda.is_available():
        return False
    return torch.cuda.get_device_capability()[0] >= 8


DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_SLSTM     = check_slstm_available()
EMBEDDING_DIM = 128

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f'Compute Capability: {major}.{minor} '
          f'({"sLSTM OK" if USE_SLSTM else "CC < 8.0 → mLSTM"})')
print(f'USE_SLSTM: {USE_SLSTM}')

# Проверка прямого прохода
test_enc  = xLSTMEncoder(
    env_train_short.observation_space,
    EMBEDDING_DIM, use_slstm=USE_SLSTM).to(DEVICE)
dummy_obs = torch.zeros(
    2, *env_train_short.observation_space.shape).to(DEVICE)
out       = test_enc(dummy_obs)
print(f'Encoder output: {out.shape}  (ожидается [2, {EMBEDDING_DIM}])')
del test_enc, dummy_obs, out

## Шаг 5: Curriculum learning — обучение в два этапа

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv

WINDOW_SIZE   = 30
BATCH_SIZE    = 32
LEARNING_RATE = 1e-4    # снижено для стабильности

policy_kwargs = dict(
    features_extractor_class=xLSTMEncoder,
    features_extractor_kwargs=dict(
        embedding_dim=EMBEDDING_DIM, num_blocks=4, use_slstm=USE_SLSTM),
    lstm_hidden_size=128,
    n_lstm_layers=1,
    enable_critic_lstm=True,
    net_arch=[],
)

# ── Этап 1: спокойный рынок 2013–2019 (300k шагов) ───────────────────────────
calm_mask = (dates_all >= '2013-01-01') & (dates_all <= '2019-12-31')

vec_env_stage1 = DummyVecEnv([lambda: MOEXTradingEnvShort(
    close_prices=close_all[calm_mask],
    features=features_all[calm_mask],
    turbulence=turb_aligned[calm_mask].values,
    turbulence_threshold=TURBULENCE_THRESHOLD,
    window_size=WINDOW_SIZE,
    margin_ratio=0.25,
)])

model_short = RecurrentPPO(
    policy='MlpLstmPolicy',
    env=vec_env_stage1,
    policy_kwargs=policy_kwargs,
    learning_rate=LEARNING_RATE,
    n_steps=2048,
    batch_size=BATCH_SIZE,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.1,     # снижено для осторожных обновлений
    ent_coef=0.02,      # больше exploration
    vf_coef=0.5,
    max_grad_norm=0.5,
    verbose=1,
    seed=42,
    tensorboard_log='./tb_logs/',
    device=DEVICE,
)

print('Этап 1: обучение на спокойном рынке 2013–2019 (300k шагов)...')
model_short.learn(total_timesteps=300_000, progress_bar=True)
print('Этап 1 завершён.')

# ── Этап 2: полный период включая кризисы (700k шагов) ───────────────────────
vec_env_stage2 = DummyVecEnv([lambda: MOEXTradingEnvShort(
    close_prices=close_train,
    features=features_train,
    turbulence=turb_train_arr,
    turbulence_threshold=TURBULENCE_THRESHOLD,
    window_size=WINDOW_SIZE,
    margin_ratio=0.25,
)])

model_short.set_env(vec_env_stage2)

print('Этап 2: дообучение на полном периоде 2009–2021 (700k шагов)...')
# reset_num_timesteps=False — продолжаем обучение не сбрасывая счётчик
model_short.learn(
    total_timesteps=700_000,
    progress_bar=True,
    reset_num_timesteps=False)

model_short.save('xlstm_ppo_moex_short_v2')
print('Обучение завершено. Итого: 1M шагов. Сохранено: xlstm_ppo_moex_short_v2')

## Шаг 6: Тестирование + метрики

In [ ]:
def run_episode(model, env):
    obs, _         = env.reset()
    obs            = obs[np.newaxis, ...]
    lstm_states    = None
    episode_starts = np.array([True])
    done           = False
    while not done:
        action, lstm_states = model.predict(
            obs, state=lstm_states,
            episode_start=episode_starts, deterministic=True)
        obs_raw, _, terminated, truncated, _ = env.step(action[0])
        obs            = obs_raw[np.newaxis, ...]
        episode_starts = np.array([terminated or truncated])
        done           = terminated or truncated
    return env.portfolio_history


def compute_metrics(portfolio_history, initial_balance, n_trades,
                    risk_free_rate=0.16):
    values  = np.array(portfolio_history, dtype=float)
    P_init  = initial_balance
    P_final = values[-1]
    CR   = (P_final - P_init) / P_init * 100
    MER  = (values.max() - P_init) / P_init * 100
    rmax = np.maximum.accumulate(values)
    MPB  = abs(((values - rmax) / (rmax + 1e-8)).min()) * 100
    APPT = (P_final - P_init) / max(n_trades, 1) / 1000
    dr   = np.diff(values) / (values[:-1] + 1e-8)
    ex   = dr - risk_free_rate / 252
    SR   = (ex.mean() / (ex.std() + 1e-8)) * np.sqrt(252)
    return {'CR': CR, 'MER': MER, 'MPB': MPB, 'APPT': APPT, 'SR': SR}


# Загружаем и тестируем
model_short_loaded = RecurrentPPO.load(
    'xlstm_ppo_moex_short_v2',
    env=DummyVecEnv([lambda: MOEXTradingEnvShort(
        close_prices=close_test, features=features_test,
        turbulence=turb_test_arr,
        turbulence_threshold=TURBULENCE_THRESHOLD,
        window_size=WINDOW_SIZE, margin_ratio=0.25,
    )])
)

print('=== ТЕСТИРОВАНИЕ (2022–2024) ===')
portfolio_hist_short = run_episode(model_short_loaded, env_test_short)

print(f'Минимальное shares_held: {env_test_short.shares_held.min():.4f}')
print(f'Были шорты: {(env_test_short.shares_held < 0).any()}')

short_metrics = compute_metrics(
    portfolio_hist_short,
    initial_balance=env_test_short.initial_balance,
    n_trades=env_test_short.trade_count,
)

print(f"""
┌─────────────────────────────────────────────────┐
│    РЕЗУЛЬТАТЫ xLSTM+PPO Short v2 (MOEX)         │
├────────────────────────────┬────────────────────┤
│ Cumulative Return (CR)     │ {short_metrics['CR']:>+.2f}%            │
│ Max Earning Rate (MER)     │ {short_metrics['MER']:>+.2f}%            │
│ Max PullBack (MPB)         │ {short_metrics['MPB']:>.2f}%             │
│ Avg Profit/Trade (APPT)    │ {short_metrics['APPT']:>.2f}k ₽          │
│ Sharpe Ratio (SR)          │ {short_metrics['SR']:>.3f}             │
│ Всего сделок               │ {env_test_short.trade_count:<20} │
└────────────────────────────┴────────────────────┘
""")

## Шаг 7: Визуализация

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

portfolio_short = np.array(portfolio_hist_short)
days            = np.arange(len(portfolio_short))
returns_pct     = (portfolio_short / env_test_short.initial_balance - 1) * 100
daily_rets      = np.diff(portfolio_short) / portfolio_short[:-1] * 100

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)
fig.suptitle(
    'xLSTM + PPO (Short v2) — MOEX (тест 2022–2024)\n'
    'Технические индикаторы + улучшенный reward + curriculum learning',
    fontsize=13, fontweight='bold', y=0.99)

ax = axes[0]
ax.plot(days, portfolio_short / 1e6, color='steelblue',
        linewidth=1.5, label='Портфель')
ax.axhline(y=env_test_short.initial_balance / 1e6, color='gray',
           linestyle='--', linewidth=1, label='Начальный капитал')
ax.set_ylabel('Стоимость (млн ₽)')
ax.set_title('Динамика портфеля')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(days, returns_pct, color='seagreen', linewidth=1.5)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.fill_between(days, returns_pct, 0,
                where=(returns_pct >= 0), alpha=0.2, color='seagreen')
ax.fill_between(days, returns_pct, 0,
                where=(returns_pct < 0), alpha=0.2, color='tomato')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylabel('Доходность (%)')
ax.set_title('Накопленная доходность')
ax.grid(True, alpha=0.3)

ax = axes[2]
colors_bar = ['seagreen' if r >= 0 else 'tomato' for r in daily_rets]
ax.bar(np.arange(len(daily_rets)), daily_rets,
       color=colors_bar, alpha=0.7, width=1.0)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlabel('Торговый день')
ax.set_ylabel('Дневная доходность (%)')
ax.set_title('Дневные доходности')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('moex_xlstm_short_v2_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('График сохранён: moex_xlstm_short_v2_results.png')

## Шаг 8: LSTM baseline (для сравнения)

In [ ]:
class LSTMEncoder(BaseFeaturesExtractor):
    def __init__(self, observation_space, hidden_dim=128):
        super().__init__(observation_space, features_dim=hidden_dim)
        _, n_features = observation_space.shape
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden_dim,
            num_layers=1, batch_first=True)

    def forward(self, obs):
        out, _ = self.lstm(obs)
        return out[:, -1, :]


vec_env_lstm_stage1 = DummyVecEnv([lambda: MOEXTradingEnvShort(
    close_prices=close_all[calm_mask],
    features=features_all[calm_mask],
    turbulence=turb_aligned[calm_mask].values,
    turbulence_threshold=TURBULENCE_THRESHOLD,
    window_size=WINDOW_SIZE, margin_ratio=0.25,
)])

baseline_model = RecurrentPPO(
    policy='MlpLstmPolicy',
    env=vec_env_lstm_stage1,
    policy_kwargs=dict(
        features_extractor_class=LSTMEncoder,
        features_extractor_kwargs=dict(hidden_dim=128),
        lstm_hidden_size=128, n_lstm_layers=1,
        enable_critic_lstm=True, net_arch=[]),
    learning_rate=LEARNING_RATE,
    n_steps=2048,
    batch_size=BATCH_SIZE,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.1,
    ent_coef=0.02,
    vf_coef=0.5,
    max_grad_norm=0.5,
    verbose=1,
    seed=42,
    device=DEVICE,
)

print('LSTM baseline — Этап 1 (300k шагов)...')
baseline_model.learn(total_timesteps=300_000, progress_bar=True)

vec_env_lstm_stage2 = DummyVecEnv([lambda: MOEXTradingEnvShort(
    close_prices=close_train, features=features_train,
    turbulence=turb_train_arr,
    turbulence_threshold=TURBULENCE_THRESHOLD,
    window_size=WINDOW_SIZE, margin_ratio=0.25,
)])
baseline_model.set_env(vec_env_lstm_stage2)

print('LSTM baseline — Этап 2 (700k шагов)...')
baseline_model.learn(
    total_timesteps=700_000,
    progress_bar=True,
    reset_num_timesteps=False)
baseline_model.save('lstm_baseline_moex_short_v2')

env_test_baseline = MOEXTradingEnvShort(
    close_prices=close_test, features=features_test,
    turbulence=turb_test_arr,
    turbulence_threshold=TURBULENCE_THRESHOLD,
    window_size=WINDOW_SIZE, margin_ratio=0.25)

baseline_hist    = run_episode(baseline_model, env_test_baseline)
baseline_metrics = compute_metrics(
    baseline_hist,
    env_test_baseline.initial_balance,
    env_test_baseline.trade_count)

# ── Итоговая таблица ──────────────────────────────────────────────────────────
print('\n' + '='*60)
print(f"{'Метрика':<10} {'LSTM+Short v2':>16} {'xLSTM+Short v2':>16}")
print('-'*60)
for k in ['CR', 'MER', 'MPB', 'APPT', 'SR']:
    print(f"{k:<10} {baseline_metrics[k]:>16.3f} {short_metrics[k]:>16.3f}")
print('='*60)